# Sim-real gap diagnostic — nearest-neighbour in raw, Fourier, and DWT space

Evaluates the structural gap between the sim and real patient populations
**without any learned encoder** — pure geometry in the input space.

Three search spaces:
- **Raw**: normalized waveform + scalar input vector (809-dim L2)
- **Fourier**: FFT magnitude of each pressure channel, first `N_FREQ` coefficients (4×N_FREQ dim)
- **DWT**: Discrete Wavelet Transform (db4, 4 levels) of each pressure channel — time-localised frequency content (~800-dim)

Plus **scalogram** visualisation (CWT Morlet) showing *where in time* and *which frequency bands*
the sim-real gap is largest for best/worst-matched patients.

Contrast all distances with the latent-space NN distances from the OT eval notebooks.

In [ ]:
import sys, h5py, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
from torch.utils.data import DataLoader

ROOT = Path(globals()['_dh'][0])
sys.path.insert(0, str(ROOT))

from dataset import load_stats, load_manifest, ReducedCVDataset, WAVE_KEYS_REDUCED

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

SIM_ROOT   = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
REAL_DATA  = Path('/home/sa4604/real_data/multibeat')
STATS_PATH = ROOT / 'norm_stats.json'

N_SIM   = 1_000_000
N_FREQ  = 30       # FFT magnitude coefficients per channel
T       = 201
CHUNK   = 50_000   # cdist chunk size — keeps GPU memory bounded

CACHE_RAW = ROOT / f'outputs/x_sim_raw_{N_SIM}.pt'
CACHE_FFT = ROOT / f'outputs/f_sim_fft{N_FREQ}_{N_SIM}.pt'

stats    = load_stats(STATS_PATH)
manifest = load_manifest(SIM_ROOT / 'manifest_train.json')
print(f'Cache paths:\n  raw: {CACHE_RAW}\n  fft: {CACHE_FFT}')

## Load real patients

In [ ]:
pas_mean = stats['waves']['Pas']['mean'];  pas_std = stats['waves']['Pas']['std'] + 1e-8
vlv_std  = stats['waves']['Vlv']['std']  + 1e-8
hr_mean  = stats['parameters']['HR']['mean']; hr_std = stats['parameters']['HR']['std'] + 1e-8
wm = torch.tensor([stats['waves'][k]['mean'] for k in WAVE_KEYS_REDUCED], dtype=torch.float32).unsqueeze(1)
ws = torch.tensor([stats['waves'][k]['std']  for k in WAVE_KEYS_REDUCED], dtype=torch.float32).unsqueeze(1) + 1e-8

wave_mean_np = wm.squeeze().numpy()
wave_std_np  = ws.squeeze().numpy()

patients = []
for fpath in sorted(REAL_DATA.glob('*.h5')):
    beats_x = []
    with h5py.File(fpath, 'r') as f:
        for bk in sorted(f.keys()):
            if not bk.startswith('beat_'): continue
            g = f[bk]
            waves = np.stack([g[f'waves/{k}'][:].astype(np.float32) for k in WAVE_KEYS_REDUCED])
            waves_t = (torch.from_numpy(waves) - wm) / ws
            sbp  = float(g['summaries/sbp'][()])
            dbp  = float(g['summaries/dbp'][()])
            map_ = float(g['summaries/map'][()])
            sv   = float(g['summaries/sv'][()])
            hr   = float(g['parameters/HR'][()])
            scalars = torch.tensor([
                (map_ - pas_mean) / pas_std, (sbp - pas_mean) / pas_std,
                (dbp  - pas_mean) / pas_std, sv / vlv_std,
                (hr   - hr_mean)  / hr_std,
            ], dtype=torch.float32)
            beats_x.append(torch.cat([waves_t.reshape(-1), scalars]))
    if beats_x:
        x_beats = torch.stack(beats_x)
        x_avg   = x_beats.mean(dim=0)
        # raw waves (unnormalised, physical units) for plotting
        raw_waves = x_avg[:4*T].view(4, T).numpy() * wave_std_np[:, None] + wave_mean_np[:, None]
        patients.append(dict(file=fpath.stem, x_avg=x_avg, raw_waves=raw_waves))

x_real = torch.stack([p['x_avg'] for p in patients])   # (60, 809)
print(f'Loaded {len(patients)} real patients  x_real: {tuple(x_real.shape)}')

## Load sim waveforms

In [ ]:
def fourier_features(x: torch.Tensor, n_freq: int = N_FREQ) -> torch.Tensor:
    """FFT magnitude of each pressure channel. x: (..., 809) → (..., 4*n_freq)"""
    waves = x[..., :4 * T].reshape(*x.shape[:-1], 4, T)
    mag   = torch.fft.rfft(waves, dim=-1).abs()[..., :n_freq]
    return mag.reshape(*x.shape[:-1], 4 * n_freq)

if CACHE_RAW.exists() and CACHE_FFT.exists():
    print('Loading from cache...')
    t0 = time.time()
    x_sim = torch.load(CACHE_RAW, map_location='cpu')
    f_sim = torch.load(CACHE_FFT, map_location='cpu')
    print(f'Loaded in {time.time()-t0:.1f}s  x_sim: {tuple(x_sim.shape)}  f_sim: {tuple(f_sim.shape)}')
else:
    print(f'Cache not found — loading {N_SIM:,} sims from HDF5...')
    t0 = time.time()
    ds     = ReducedCVDataset(str(SIM_ROOT / 'train'), manifest['index'][:N_SIM], stats)
    loader = DataLoader(ds, batch_size=2048, shuffle=False, num_workers=8)
    x_list = []
    loaded = 0
    for _, x_b in loader:
        x_list.append(x_b)
        loaded += len(x_b)
        print(f'  {loaded:,}/{N_SIM:,}', end='\r', flush=True)
    ds.close()
    x_sim = torch.cat(x_list)
    print(f'\nLoaded in {time.time()-t0:.1f}s  x_sim: {tuple(x_sim.shape)}  ({x_sim.nbytes/1e9:.2f} GB)')

    print('Computing Fourier features in chunks...')
    t1 = time.time()
    f_chunks = []
    for i in range(0, len(x_sim), CHUNK):
        f_chunks.append(fourier_features(x_sim[i:i+CHUNK].to(device)).cpu())
    f_sim = torch.cat(f_chunks)
    print(f'FFT done in {time.time()-t1:.1f}s  f_sim: {tuple(f_sim.shape)}  ({f_sim.nbytes/1e9:.2f} GB)')

    print(f'Saving caches...')
    torch.save(x_sim, CACHE_RAW)
    torch.save(f_sim, CACHE_FFT)
    print(f'  saved {CACHE_RAW.name}  ({CACHE_RAW.stat().st_size/1e9:.2f} GB)')
    print(f'  saved {CACHE_FFT.name}  ({CACHE_FFT.stat().st_size/1e9:.2f} GB)')

## Nearest-neighbour search — raw input space (809-dim L2)

In [ ]:
print('Computing NN in raw space (chunked cdist, x_sim stays on CPU)...')
t0 = time.time()
x_real_gpu = x_real.to(device)   # (60, 809) — tiny

nn_dist_raw = torch.full((len(patients),), float('inf'))
nn_idx_raw  = torch.zeros(len(patients), dtype=torch.long)

for start in range(0, len(x_sim), CHUNK):
    end  = min(start + CHUNK, len(x_sim))
    d    = torch.cdist(x_real_gpu, x_sim[start:end].to(device))  # (60, CHUNK)
    dmin, imin = d.min(dim=1)
    update = dmin.cpu() < nn_dist_raw
    nn_dist_raw[update] = dmin.cpu()[update]
    nn_idx_raw[update]  = (imin.cpu() + start)[update]

print(f'Done in {time.time()-t0:.1f}s')

rand_i      = torch.randperm(len(x_sim))[:200]
sim_sim_raw = torch.cdist(x_sim[rand_i[:100]].to(device), x_sim[rand_i[100:]].to(device)).flatten().cpu()

print(f'\nRaw space (809-dim L2):')
print(f'  sim-sim random : mean={sim_sim_raw.mean():.3f}  median={sim_sim_raw.median():.3f}')
print(f'  real → NN sim  : mean={nn_dist_raw.mean():.3f}  median={nn_dist_raw.median():.3f}  '
      f'min={nn_dist_raw.min():.3f}  max={nn_dist_raw.max():.3f}')
print(f'  ratio          : {(nn_dist_raw.mean() / sim_sim_raw.mean()).item():.2f}x sim-sim')

## Nearest-neighbour search — Fourier space

FFT magnitude of each of the 4 pressure waveform channels, first `N_FREQ` coefficients.
Captures rhythm and morphology, invariant to phase shifts within a beat.

In [ ]:
print(f'Moving f_sim to GPU ({f_sim.nbytes/1e9:.2f} GB)...')
f_sim_gpu  = f_sim.to(device)
f_real_gpu = fourier_features(x_real_gpu)   # (60, 4*N_FREQ)
print(f'f_real: {tuple(f_real_gpu.shape)}  f_sim: {tuple(f_sim_gpu.shape)}')

print('Computing NN in Fourier space...')
t0 = time.time()
# f_sim fits on GPU (480 MB) — one-shot cdist
dists_fft   = torch.cdist(f_real_gpu, f_sim_gpu)   # (60, 1M)
nn_dist_fft, nn_idx_fft = dists_fft.min(dim=1)
nn_dist_fft = nn_dist_fft.cpu()
nn_idx_fft  = nn_idx_fft.cpu()
print(f'Done in {time.time()-t0:.1f}s')

sim_sim_fft = torch.cdist(f_sim_gpu[rand_i[:100]], f_sim_gpu[rand_i[100:]]).flatten().cpu()

print(f'\nFourier space ({4*N_FREQ}-dim L2):')
print(f'  sim-sim random : mean={sim_sim_fft.mean():.3f}  median={sim_sim_fft.median():.3f}')
print(f'  real → NN sim  : mean={nn_dist_fft.mean():.3f}  median={nn_dist_fft.median():.3f}  '
      f'min={nn_dist_fft.min():.3f}  max={nn_dist_fft.max():.3f}')
print(f'  ratio          : {(nn_dist_fft.mean() / sim_sim_fft.mean()).item():.2f}x sim-sim')

## Nearest-neighbour search — DWT space

Discrete Wavelet Transform (db4, 4 levels) on the 4 pressure channels.
Captures time-localised frequency content that a global FFT misses — e.g. early systolic spikes or end-diastolic dip differences.
Features are concatenated coefficients across all levels for each channel (~800-dim).
Cache stored as float16 to halve disk use; loaded back as float32 for cdist.

In [ ]:
import pywt

N_DWT_LEVELS = 4
DWT_WAVELET  = 'db4'
CACHE_DWT    = ROOT / f'outputs/d_sim_dwt_{DWT_WAVELET}_{N_DWT_LEVELS}_{N_SIM}.pt'

def dwt_features(x: np.ndarray) -> np.ndarray:
    """DWT of 4 pressure channels. x: (B, 809) numpy → (B, 4*D_dwt)"""
    waves = x[:, :4*T].reshape(len(x), 4, T)          # (B, 4, T)
    parts = []
    for ci in range(4):
        coeffs = pywt.wavedec(waves[:, ci, :], DWT_WAVELET, level=N_DWT_LEVELS, axis=-1)
        parts.append(np.concatenate(coeffs, axis=-1))  # (B, D_level)
    return np.concatenate(parts, axis=-1)               # (B, 4*D_level)

# DWT features for real patients (fast — 60 rows)
d_real  = dwt_features(x_real.numpy())                 # (60, DWT_DIM)
DWT_DIM = d_real.shape[1]
print(f'DWT feature dim: {DWT_DIM}')

if CACHE_DWT.exists():
    print(f'Loading DWT cache...')
    t0    = time.time()
    d_sim = torch.load(CACHE_DWT, map_location='cpu').float()
    print(f'Loaded in {time.time()-t0:.1f}s  d_sim: {tuple(d_sim.shape)}')
else:
    print(f'Computing DWT features for {N_SIM:,} sims (CPU, chunked)...')
    t0       = time.time()
    d_chunks = []
    for start in range(0, len(x_sim), CHUNK):
        end = min(start + CHUNK, len(x_sim))
        d_chunks.append(torch.from_numpy(dwt_features(x_sim[start:end].numpy())).half())
        if (start // CHUNK) % 10 == 0:
            print(f'  {start:,}/{N_SIM:,}', end='\r', flush=True)
    d_sim = torch.cat(d_chunks)
    print(f'\nDWT done in {time.time()-t0:.1f}s  d_sim: {tuple(d_sim.shape)}')
    torch.save(d_sim, CACHE_DWT)
    print(f'Saved {CACHE_DWT.name}  ({CACHE_DWT.stat().st_size/1e9:.2f} GB)')
    d_sim = d_sim.float()

# NN search in DWT space (chunked cdist)
print('Computing NN in DWT space (chunked cdist)...')
t0          = time.time()
d_real_gpu  = torch.from_numpy(d_real).to(device)
nn_dist_dwt = torch.full((len(patients),), float('inf'))
nn_idx_dwt  = torch.zeros(len(patients), dtype=torch.long)

for start in range(0, len(d_sim), CHUNK):
    end  = min(start + CHUNK, len(d_sim))
    dmat = torch.cdist(d_real_gpu, d_sim[start:end].to(device))
    dmin, imin = dmat.min(dim=1)
    update = dmin.cpu() < nn_dist_dwt
    nn_dist_dwt[update] = dmin.cpu()[update]
    nn_idx_dwt[update]  = (imin.cpu() + start)[update]

print(f'Done in {time.time()-t0:.1f}s')

sim_sim_dwt = torch.cdist(d_sim[rand_i[:100]].to(device), d_sim[rand_i[100:]].to(device)).flatten().cpu()

print(f'\nDWT space ({DWT_DIM}-dim L2):')
print(f'  sim-sim random : mean={sim_sim_dwt.mean():.3f}  median={sim_sim_dwt.median():.3f}')
print(f'  real → NN sim  : mean={nn_dist_dwt.mean():.3f}  median={nn_dist_dwt.median():.3f}  '
      f'min={nn_dist_dwt.min():.3f}  max={nn_dist_dwt.max():.3f}')
print(f'  ratio          : {(nn_dist_dwt.mean() / sim_sim_dwt.mean()).item():.2f}x sim-sim')

## Scalogram comparison — real vs nearest DWT-space sim (CWT)

Continuous Wavelet Transform (complex Morlet) gives a 2D time-frequency view of each channel.
Colour shows log₁₀(real power / sim power): red = real stronger, blue = sim stronger, white = matched.
Reveals *where in time* and *which frequency bands* drive the sim-real gap.

In [ ]:
import pywt

SCALES  = np.arange(2, 80, 2)       # 39 scales — coarse to fine
WAVELET = 'cmor1.5-1.0'             # complex Morlet
FREQS   = pywt.scale2frequency(WAVELET, SCALES) * T

def plot_scalograms(patient_indices, nn_indices, dist_arr, title):
    n = len(patient_indices)
    fig, axes = plt.subplots(n, 4, figsize=(20, 3*n))
    if n == 1: axes = axes[None, :]
    im = None
    for row, (pi, ni) in enumerate(zip(patient_indices, nn_indices)):
        real_waves = patients[pi]['raw_waves']
        sim_norm   = x_sim[ni, :4*T].view(4, T).numpy()
        sim_waves  = sim_norm * wave_std_np[:, None] + wave_mean_np[:, None]
        for ci, ch in enumerate(WAVE_KEYS_REDUCED):
            real_pwr = np.abs(pywt.cwt(real_waves[ci], SCALES, WAVELET)[0])
            sim_pwr  = np.abs(pywt.cwt(sim_waves[ci],  SCALES, WAVELET)[0])
            eps      = 1e-6
            log_ratio = np.log10((real_pwr + eps) / (sim_pwr + eps))
            vmax = np.abs(log_ratio).max()
            ax = axes[row, ci]
            im = ax.imshow(log_ratio, aspect='auto', origin='lower', cmap='RdBu_r',
                           vmin=-vmax, vmax=vmax, extent=[0, T, FREQS[-1], FREQS[0]])
            ax.set_title(f'{patients[pi]["file"]} — {ch}\ndist={dist_arr[pi]:.2f}', fontsize=8)
            ax.set_xlabel('Time step', fontsize=7)
            ax.set_ylabel('Freq (norm×T)', fontsize=7)
            ax.tick_params(labelsize=6)
    if im is not None:
        fig.colorbar(im, ax=axes.ravel().tolist(), label='log₁₀(real/sim power)', shrink=0.5)
    plt.suptitle(f'Scalogram difference (CWT Morlet) — {title}', fontsize=11)
    plt.tight_layout(); plt.show()

sorted_by_dwt = np.argsort(nn_dist_dwt.numpy())
best2_dwt     = sorted_by_dwt[:2].tolist()
worst2_dwt    = sorted_by_dwt[-2:].tolist()

print('=== Best 2 matches (DWT space) ===')
plot_scalograms(best2_dwt,  nn_idx_dwt[best2_dwt].tolist(),  nn_dist_dwt.numpy(), 'best DWT matches')
print('=== Worst 2 matches (DWT space) ===')
plot_scalograms(worst2_dwt, nn_idx_dwt[worst2_dwt].tolist(), nn_dist_dwt.numpy(), 'worst DWT matches')

## Distance comparison — raw vs Fourier vs DWT (normalised by sim-sim median)

In [ ]:
# Normalise by sim-sim median so distances are comparable across spaces
raw_norm = nn_dist_raw.numpy() / sim_sim_raw.median().item()
fft_norm = nn_dist_fft.numpy() / sim_sim_fft.median().item()
dwt_norm = nn_dist_dwt.numpy() / sim_sim_dwt.median().item()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: histograms — all three spaces
ax = axes[0]
ax.hist(raw_norm, bins=20, alpha=0.6, color='steelblue',
        label=f'Raw 809-dim  (mean={raw_norm.mean():.2f}x)')
ax.hist(fft_norm, bins=20, alpha=0.6, color='mediumseagreen',
        label=f'Fourier {4*N_FREQ}-dim (mean={fft_norm.mean():.2f}x)')
ax.hist(dwt_norm, bins=20, alpha=0.6, color='orchid',
        label=f'DWT {DWT_DIM}-dim (mean={dwt_norm.mean():.2f}x)')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='sim-sim median')
ax.set_xlabel('NN distance / sim-sim median')
ax.set_title('Real → NN sim distance (normalised)')
ax.legend(fontsize=8)

# Panel 2: per-patient scatter — raw vs Fourier
ax = axes[1]
ax.scatter(raw_norm, fft_norm, s=40, alpha=0.8, color='mediumseagreen')
lim = max(raw_norm.max(), fft_norm.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8)
ax.set_xlabel('Normalised dist — raw space')
ax.set_ylabel('Normalised dist — Fourier space')
ax.set_title('Raw vs Fourier per patient')

# Panel 3: per-patient scatter — raw vs DWT
ax = axes[2]
ax.scatter(raw_norm, dwt_norm, s=40, alpha=0.8, color='orchid')
lim = max(raw_norm.max(), dwt_norm.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8)
ax.set_xlabel('Normalised dist — raw space')
ax.set_ylabel('Normalised dist — DWT space')
ax.set_title('Raw vs DWT per patient')

plt.suptitle('Sim-real gap: NN distances in raw / Fourier / DWT space', fontsize=11)
plt.tight_layout(); plt.show()

# Pairwise NN agreement
same_raw_fft = (nn_idx_raw == nn_idx_fft).sum().item()
same_raw_dwt = (nn_idx_raw == nn_idx_dwt).sum().item()
same_fft_dwt = (nn_idx_fft == nn_idx_dwt).sum().item()
N = len(patients)
print(f'NN agreement (same nearest sim out of {N} patients):')
print(f'  raw ↔ Fourier : {same_raw_fft}/{N} ({100*same_raw_fft/N:.0f}%)')
print(f'  raw ↔ DWT     : {same_raw_dwt}/{N} ({100*same_raw_dwt/N:.0f}%)')
print(f'  Fourier ↔ DWT : {same_fft_dwt}/{N} ({100*same_fft_dwt/N:.0f}%)')

## Waveform overlay — real vs nearest sim

Visually assess the match quality. Shows best and worst patients by raw-space NN distance.

In [ ]:
def plot_overlay(patient_indices, nn_indices, space_label, dist_arr):
    n = len(patient_indices)
    fig, axes = plt.subplots(n, 4, figsize=(18, 3 * n))
    if n == 1: axes = axes[None, :]

    for row, (pi, ni) in enumerate(zip(patient_indices, nn_indices)):
        pat       = patients[pi]
        real_phys = pat['raw_waves']                       # (4, T) in mmHg

        # Reconstruct sim waveform from normalised input vector
        sim_norm  = x_sim[ni, :4*T].view(4, T).numpy()    # normalised
        sim_phys  = sim_norm * wave_std_np[:, None] + wave_mean_np[:, None]

        for ci, ch in enumerate(WAVE_KEYS_REDUCED):
            ax = axes[row, ci]
            ax.plot(real_phys[ci], color='steelblue',    linewidth=1.5, label='real')
            ax.plot(sim_phys[ci],  color='tomato',       linewidth=1.5, linestyle='--', label='nearest sim')
            ax.set_title(f'{pat["file"]} — {ch}\ndist={dist_arr[pi]:.2f}', fontsize=8)
            ax.set_ylabel('mmHg', fontsize=7); ax.tick_params(labelsize=6)
            if row == 0 and ci == 0: ax.legend(fontsize=7)

    fig.suptitle(f'Real vs nearest sim — {space_label} NN', fontsize=11)
    plt.tight_layout(); plt.show()

# Best 3 and worst 3 patients by raw NN distance
sorted_by_raw = np.argsort(nn_dist_raw.numpy())
best3  = sorted_by_raw[:3].tolist()
worst3 = sorted_by_raw[-3:].tolist()

print('=== Best 3 matches (raw space) ===')
plot_overlay(best3,  nn_idx_raw[best3].tolist(),  'raw (best)',  nn_dist_raw.numpy())

print('=== Worst 3 matches (raw space) ===')
plot_overlay(worst3, nn_idx_raw[worst3].tolist(), 'raw (worst)', nn_dist_raw.numpy())

## Waveform overlay — Fourier-space NN (best and worst)

In [ ]:
sorted_by_fft = np.argsort(nn_dist_fft.numpy())
best3_fft  = sorted_by_fft[:3].tolist()
worst3_fft = sorted_by_fft[-3:].tolist()

print('=== Best 3 matches (Fourier space) ===')
plot_overlay(best3_fft,  nn_idx_fft[best3_fft].tolist(),  'Fourier (best)',  nn_dist_fft.numpy())

print('=== Worst 3 matches (Fourier space) ===')
plot_overlay(worst3_fft, nn_idx_fft[worst3_fft].tolist(), 'Fourier (worst)', nn_dist_fft.numpy())

## Waveform overlay — Fourier-space NN (where it disagrees with raw)

In [ ]:
# Patients where raw and Fourier NNs disagree — most informative cases
disagree = torch.where(nn_idx_raw != nn_idx_fft)[0].tolist()
print(f'{len(disagree)} patients with different raw vs Fourier NN')

if disagree:
    show = disagree[:4]
    fig, axes = plt.subplots(len(show), 4, figsize=(18, 3 * len(show)))
    if len(show) == 1: axes = axes[None, :]

    for row, pi in enumerate(show):
        pat = patients[pi]
        real_phys = pat['raw_waves']

        sim_raw_phys = x_sim[nn_idx_raw[pi], :4*T].view(4, T).numpy() * wave_std_np[:, None] + wave_mean_np[:, None]
        sim_fft_phys = x_sim[nn_idx_fft[pi], :4*T].view(4, T).numpy() * wave_std_np[:, None] + wave_mean_np[:, None]

        for ci, ch in enumerate(WAVE_KEYS_REDUCED):
            ax = axes[row, ci]
            ax.plot(real_phys[ci],    color='steelblue',      linewidth=1.5, label='real')
            ax.plot(sim_raw_phys[ci], color='tomato',         linewidth=1.5, linestyle='--', label=f'raw NN (d={nn_dist_raw[pi]:.2f})')
            ax.plot(sim_fft_phys[ci], color='mediumseagreen', linewidth=1.5, linestyle=':',  label=f'Fourier NN (d={nn_dist_fft[pi]:.2f})')
            ax.set_title(f'{pat["file"]} — {ch}', fontsize=8)
            ax.set_ylabel('mmHg', fontsize=7); ax.tick_params(labelsize=6)
            if row == 0 and ci == 0: ax.legend(fontsize=7)

    plt.suptitle('Raw vs Fourier NN disagreement cases — which is a better match?', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('Raw and Fourier NNs agree for all patients.')

## Spectral comparison — real vs nearest sim (average across patients)

In [ ]:
freqs = np.fft.rfftfreq(T)[:N_FREQ]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ci, ch in enumerate(WAVE_KEYS_REDUCED):
    ax = axes[ci]

    real_specs, raw_specs, fft_specs = [], [], []
    for pi, pat in enumerate(patients):
        real_wave  = torch.from_numpy(pat['raw_waves'][ci])   # (T,) physical
        sim_raw    = x_sim[nn_idx_raw[pi], ci*T:(ci+1)*T]     # normalised
        sim_fft    = x_sim[nn_idx_fft[pi], ci*T:(ci+1)*T]

        real_specs.append(torch.fft.rfft(real_wave).abs().numpy()[:N_FREQ])
        raw_specs.append(torch.fft.rfft(sim_raw).abs().numpy()[:N_FREQ])
        fft_specs.append(torch.fft.rfft(sim_fft).abs().numpy()[:N_FREQ])

    real_mean = np.mean(real_specs, axis=0)
    raw_mean  = np.mean(raw_specs,  axis=0)
    fft_mean  = np.mean(fft_specs,  axis=0)

    ax.plot(freqs, real_mean, color='steelblue',      linewidth=2,   label='real (mean)')
    ax.plot(freqs, raw_mean,  color='tomato',         linewidth=1.5, linestyle='--', label='raw NN (mean)')
    ax.plot(freqs, fft_mean,  color='mediumseagreen', linewidth=1.5, linestyle=':',  label='Fourier NN (mean)')
    ax.set_title(f'{ch} spectrum', fontsize=9)
    ax.set_xlabel('Normalised frequency')
    ax.set_ylabel('FFT magnitude')
    if ci == 0: ax.legend(fontsize=8)

plt.suptitle('Mean FFT magnitude — real vs raw NN sim vs Fourier NN sim', fontsize=11)
plt.tight_layout(); plt.show()

## Summary

| Space | Dim | Real→NN mean | Sim-sim mean | Ratio |
|---|---|---|---|---|
| Raw | 809 | — | — | — |
| Fourier | 4×N_FREQ | — | — | — |
| DWT | 4×D_dwt | — | — | — |

*(Fill in from cell outputs above)*

Compare ratios to the OT latent-space ratio from `exp_cnn4e64-ae-reduced-maf-joint_1M_ot-sinkhorn_eval_v2.ipynb`
to see how much the learned transport closes the gap vs raw / Fourier / DWT geometry.

**Scalogram interpretation**: RdBu_r difference scalogram — red regions = frequency bands where real patients
have more power than the nearest sim; blue = sim has more power. Persistent patterns across patients
point to systematic waveform features the simulator under- or over-represents.